# LlamaIndex Practice

Welcome to your hands-on practice with LlamaIndex! In this notebook, you'll learn how to:
- Create custom tools for AI agents
- Build a functional agent using Google Gemini
- Integrate Model Context Protocol (MCP) tools
- Implement observability and debugging
- Complete a creative challenge to apply your knowledge

## What You'll Build

In this notebook, you'll create an intelligent **Travel Planning Assistant** that can:
- Search for flight information
- Get weather forecasts for destinations
- Convert currencies for travel budgeting
- Use Model Context Protocol (MCP) for enhanced capabilities

By the end, you'll understand the complete agent creation flow with LlamaIndex and be ready to build your own creative agents!

## Step 1: Initial Setup & Installation

First, let's install the required packages. We need:
- **llama-index-llms-openai**: OpenAI integration
- **llama-index**: Core LlamaIndex framework
- **llama-index-tools-mcp**: Model Context Protocol tools support

In [31]:
%pip install llama-index-llms-openai llama-index llama-index-tools-mcp -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 2: Configure API Key

You'll need an OpenAI API key to use GPT-4.1-mini. You also need to provide the base URL.

In [32]:
import getpass
import os
from dotenv import load_dotenv

load_dotenv()

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key: ")

if "OPENAI_BASE_URL" not in os.environ:
    os.environ["OPENAI_BASE_URL"] = getpass.getpass("Enter your OpenAI Base URL: ")

## Step 3: Create Custom Tools

Tools are functions that your agent can call to perform specific tasks. Let's create travel-related tools.

**Key Concepts:**
- Each tool is a Python function with clear docstrings
- Type hints help the LLM understand parameter types
- Docstrings describe what the tool does (the LLM reads this!)

In [33]:
from datetime import datetime, timedelta
import random


def search_flights(origin: str, destination: str, date: str) -> str:
    """Search for available flights between two cities on a specific date.

    Args:
        origin: The departure city (e.g., 'New York')
        destination: The arrival city (e.g., 'Paris')
        date: The travel date in YYYY-MM-DD format

    Returns:
        A string with flight information including price and duration
    """
    # Simulate flight search
    flight_number = f"FL{random.randint(100, 999)}"
    price = random.randint(300, 1500)
    duration = random.randint(4, 15)

    return f"Found flight {flight_number} from {origin} to {destination} on {date}. Price: ${price}, Duration: {duration}h"


def get_weather(city: str, days_ahead: int = 0) -> str:
    """Get weather forecast for a city.

    Args:
        city: The city name to get weather for
        days_ahead: Number of days ahead to forecast (0 = today, 1 = tomorrow, etc.)

    Returns:
        A string describing the weather conditions
    """
    conditions = ["Sunny", "Partly Cloudy", "Cloudy", "Rainy", "Stormy"]
    condition = random.choice(conditions)
    temp = random.randint(15, 30)
    forecast_date = datetime.now() + timedelta(days=days_ahead)

    return f"Weather in {city} on {forecast_date.strftime('%Y-%m-%d')}: {condition}, {temp}°C"


def convert_currency(amount: float, from_currency: str, to_currency: str) -> str:
    """Convert an amount from one currency to another.

    Args:
        amount: The amount to convert
        from_currency: Source currency code (e.g., 'USD')
        to_currency: Target currency code (e.g., 'EUR')

    Returns:
        A string with the converted amount
    """
    # Simplified exchange rates (in practice, you'd use a real API)
    rates = {
        ("USD", "EUR"): 0.92,
        ("USD", "GBP"): 0.79,
        ("USD", "JPY"): 149.50,
        ("EUR", "USD"): 1.09,
        ("GBP", "USD"): 1.27,
    }

    rate = rates.get((from_currency, to_currency), 1.0)
    converted = amount * rate

    return f"{amount} {from_currency} = {converted:.2f} {to_currency}"

## Step 4: Initialize the Language Model

Now let's set up Google Gemini as our LLM. We're using **gemini-2.5-flash** for fast, efficient responses.

In [34]:
from llama_index.llms.openai import OpenAI

llm = OpenAI(
    model="gpt-4.1-mini",
    api_key=os.environ["OPENAI_API_KEY"],
    api_base=os.environ["OPENAI_BASE_URL"]
)

## Step 4: Initialize the Language Model

Now let's set up OpenAI GPT-4.1-mini as our LLM.

In [35]:
from llama_index.core.agent.workflow import FunctionAgent

agent = FunctionAgent(
    tools=[search_flights, get_weather, convert_currency],
    llm=llm,
    verbose=True  # This helps us see what the agent is doing
)

## Step 6: Create an Observability Helper

Before testing our agent, let's create a helper function that gives us **detailed visibility** into what the agent is doing.

**Why Observability Matters:**
- 🔍 See which tools the agent chooses
- 📊 Inspect the inputs passed to each tool
- ✅ Verify the outputs returned
- 🐛 Debug issues when the agent doesn't behave as expected

The `run_agent_verbose()` function will:
1. Display the user's query
2. Stream events as the agent works
3. Print details about each tool call
4. Return the final response

This is crucial for understanding and debugging agent behavior!

In [36]:
from llama_index.core.agent.workflow import ToolCallResult


async def run_agent_verbose(query: str):
    """Run the agent and print detailed information about tool calls."""
    print(f"🤖 User Query: {query}\n")
    print("=" * 60)

    handler = agent.run(query)
    async for event in handler.stream_events():
        if isinstance(event, ToolCallResult):
            print(f"🔧 Tool Used: {event.tool_name}")
            print(f"📥 Input: {event.tool_kwargs}")
            print(f"📤 Output: {event.tool_output}")
            print("-" * 60)

    result = await handler
    print(f"\n✨ Final Response:\n{result}\n")
    return result

## Step 7: Test Your Agent with Simple Queries

Let's start with basic queries to see how the agent uses tools:

In [37]:
response = await run_agent_verbose(
    "What's the weather like in Tokyo today?"
)

🤖 User Query: What's the weather like in Tokyo today?

🔧 Tool Used: get_weather
📥 Input: {'city': 'Tokyo', 'days_ahead': 0}
📤 Output: Weather in Tokyo on 2026-02-26: Cloudy, 29°C
------------------------------------------------------------

✨ Final Response:
The weather in Tokyo today is cloudy with a temperature of 29°C.



In [38]:
# Inspect which tools were called
print("Tool Calls Made:")
for tool_call in response.tool_calls:
    print(f"  - {tool_call}")

Tool Calls Made:
  - tool_name='get_weather' tool_kwargs={'city': 'Tokyo', 'days_ahead': 0} tool_id='call_YV5XFBMhGpOlqGOqaiPpMk13' tool_output=ToolOutput(blocks=[TextBlock(block_type='text', text='Weather in Tokyo on 2026-02-26: Cloudy, 29°C')], tool_name='get_weather', raw_input={'args': (), 'kwargs': {'city': 'Tokyo', 'days_ahead': 0}}, raw_output='Weather in Tokyo on 2026-02-26: Cloudy, 29°C', is_error=False) return_direct=False


## Step 8: Complex Multi-Tool Query

Watch how the agent chains multiple tools together to answer complex questions:

In [39]:
response = await run_agent_verbose(
    "I want to fly from London to Barcelona on 2025-12-15. What's the weather there, and how much is 500 USD in EUR?"
)

🤖 User Query: I want to fly from London to Barcelona on 2025-12-15. What's the weather there, and how much is 500 USD in EUR?

🔧 Tool Used: search_flights
📥 Input: {'origin': 'London', 'destination': 'Barcelona', 'date': '2025-12-15'}
📤 Output: Found flight FL550 from London to Barcelona on 2025-12-15. Price: $557, Duration: 12h
------------------------------------------------------------
🔧 Tool Used: get_weather
📥 Input: {'city': 'Barcelona'}
📤 Output: Weather in Barcelona on 2026-02-26: Partly Cloudy, 23°C
------------------------------------------------------------
🔧 Tool Used: convert_currency
📥 Input: {'amount': 500, 'from_currency': 'USD', 'to_currency': 'EUR'}
📤 Output: 500 USD = 460.00 EUR
------------------------------------------------------------

✨ Final Response:
For your flight from London to Barcelona on 2025-12-15, there is a flight FL550 available with a price of $557 and a duration of 12 hours.

The current weather in Barcelona is partly cloudy with a temperature of 2

## Step 9: Managing Context & Memory

By default, `.run()` is stateless (doesn't remember previous conversations). Let's add memory using a **Context** object:

In [40]:
from llama_index.core.workflow import Context

# Create a new agent instance for this demo
memory_agent = FunctionAgent(
    tools=[search_flights, get_weather, convert_currency],
    llm=llm
)

# Create a context to maintain conversation history
ctx = Context(memory_agent)

# First message - introduce yourself
response1 = await memory_agent.run(
    "Hi! I'm planning a trip to Rome in December. My name is Alex.",
    ctx=ctx
)
print(f"Response 1: {response1}\n")

# Second message - agent should remember your name
response2 = await memory_agent.run(
    "What's my name, and what's the weather like in my destination?",
    ctx=ctx
)
print(f"Response 2: {response2}")

Response 1: Hi Alex! That sounds like a wonderful trip. How can I assist you with your travel plans to Rome in December? Are you looking for flights, accommodation, weather information, or something else?

Response 2: Your name is Alex. The current weather in Rome is sunny with a temperature of around 16°C. If you want, I can also check the weather forecast for a specific date in December. Would you like me to do that?


## Step 10: Advanced - Model Context Protocol (MCP) Tools

MCP is a standardized protocol for connecting AI models to external tools and data sources. Let's integrate an MCP tool!

**What is MCP?**
- Open protocol for AI-to-tool communication
- Enables standardized tool integration
- Supports local and remote tools

### Creating a Custom MCP Tool

Let's create an MCP-compatible tool that fetches travel tips:

In [41]:
%pip install mcp httpx -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [42]:
from llama_index.core.tools import FunctionTool


def get_travel_tips(destination: str) -> str:
    """Get essential travel tips for a destination.

    Args:
        destination: The city or country to get tips for

    Returns:
        A string with helpful travel tips
    """
    tips_database = {
        "paris": "🇫🇷 Tips for Paris: Try croissants at local boulangeries, use the Metro, visit Eiffel Tower early morning",
        "tokyo": "🇯🇵 Tips for Tokyo: Get a Suica card for transport, try ramen at local shops, visit temples in Asakusa",
        "new york": "🇺🇸 Tips for NYC: Use subway, try pizza and bagels, visit Central Park and Times Square",
        "barcelona": "🇪🇸 Tips for Barcelona: Visit Sagrada Familia, try tapas and paella, walk Las Ramblas",
        "rome": "🇮🇹 Tips for Rome: Visit Colosseum early, try authentic carbonara, throw coin in Trevi Fountain",
    }

    city_lower = destination.lower()
    for key in tips_database:
        if key in city_lower:
            return tips_database[key]

    return f"💡 General tips: Research local customs, learn basic phrases, try local cuisine!"


# Convert to LlamaIndex tool
travel_tips_tool = FunctionTool.from_defaults(fn=get_travel_tips)

# Create agent with MCP-style tool
mcp_agent = FunctionAgent(
    tools=[search_flights, get_weather, convert_currency, travel_tips_tool],
    llm=llm
)

print("✅ Agent with MCP tool created!")

✅ Agent with MCP tool created!


### Testing the Enhanced Agent

In [43]:
async def run_mcp_agent(query: str):
    """Run the MCP-enhanced agent."""
    print(f"🤖 Query: {query}\n")
    handler = mcp_agent.run(query)
    async for event in handler.stream_events():
        if isinstance(event, ToolCallResult):
            print(f"🔧 {event.tool_name} → {event.tool_output}")

    result = await handler
    print(f"\n✨ {result}\n")
    return result


response = await run_mcp_agent(
    "I'm planning a trip to Tokyo. Give me travel tips and check the weather for tomorrow."
)

🤖 Query: I'm planning a trip to Tokyo. Give me travel tips and check the weather for tomorrow.

🔧 get_travel_tips → 🇯🇵 Tips for Tokyo: Get a Suica card for transport, try ramen at local shops, visit temples in Asakusa
🔧 get_weather → Weather in Tokyo on 2026-02-27: Partly Cloudy, 21°C

✨ Here are some travel tips for Tokyo: Get a Suica card for convenient transport, try ramen at local shops, and visit temples in Asakusa.

The weather in Tokyo tomorrow is expected to be partly cloudy with a temperature around 21°C. If you need more information or help with your trip, feel free to ask!



## Step 11: Enhanced Observability with Custom Callbacks

Let's implement more detailed observability to track agent performance:

In [44]:
import time
from typing import List


class AgentObserver:
    """Track and display agent performance metrics."""

    def __init__(self):
        self.tool_calls = []
        self.start_time = None
        self.end_time = None

    async def observe_agent(self, query: str, agent: FunctionAgent):
        """Run agent and collect observability data."""
        self.tool_calls = []
        self.start_time = time.time()

        print(f"🎯 Query: {query}\n")
        print("📊 Observability Dashboard")
        print("=" * 60)

        handler = agent.run(query)
        async for event in handler.stream_events():
            if isinstance(event, ToolCallResult):
                self.tool_calls.append({
                    'tool': event.tool_name,
                    'input': event.tool_kwargs,
                    'output': event.tool_output
                })
                print(f"⚙️  Tool Call #{len(self.tool_calls)}: {event.tool_name}")

        result = await handler
        self.end_time = time.time()

        # Display metrics
        print("\n" + "=" * 60)
        print("📈 Performance Metrics:")
        print(f"  ⏱️  Total Time: {self.end_time - self.start_time:.2f}s")
        print(f"  🔧 Tools Called: {len(self.tool_calls)}")
        print(f"  📝 Tool Names: {[t['tool'] for t in self.tool_calls]}")
        print(f"\n💬 Response: {result}")

        return result


# Test the observer
observer = AgentObserver()
await observer.observe_agent(
    "Find flights from Paris to Tokyo on 2025-12-20, check weather, and convert 1000 EUR to JPY",
    mcp_agent
)

🎯 Query: Find flights from Paris to Tokyo on 2025-12-20, check weather, and convert 1000 EUR to JPY

📊 Observability Dashboard
⚙️  Tool Call #1: search_flights
⚙️  Tool Call #2: get_weather
⚙️  Tool Call #3: convert_currency

📈 Performance Metrics:
  ⏱️  Total Time: 2.92s
  🔧 Tools Called: 3
  📝 Tool Names: ['search_flights', 'get_weather', 'convert_currency']

💬 Response: Here is the information you requested:

- Flight from Paris to Tokyo on 2025-12-20: Flight FL178, Price: $1114, Duration: 15 hours.
- Weather in Tokyo today: Rainy, 23°C.
- Currency conversion: 1000 EUR = 1000.00 JPY.

If you need any more details or assistance, feel free to ask!


AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='Here is the information you requested:\n\n- Flight from Paris to Tokyo on 2025-12-20: Flight FL178, Price: $1114, Duration: 15 hours.\n- Weather in Tokyo today: Rainy, 23°C.\n- Currency conversion: 1000 EUR = 1000.00 JPY.\n\nIf you need any more details or assistance, feel free to ask!')]), structured_response=None, current_agent_name='Agent', raw={'id': 'chatcmpl-DDSsy1kItnpu8pYYjagecEt0qh3KI', 'choices': [{'delta': {'content': None, 'function_call': None, 'refusal': None, 'role': None, 'tool_calls': None}, 'finish_reason': 'stop', 'index': 0, 'logprobs': None, 'content_filter_results': {}}], 'created': 1772101672, 'model': 'gpt-4.1-mini-2025-04-14', 'object': 'chat.completion.chunk', 'service_tier': None, 'system_fingerprint': 'fp_b6f445fc1c', 'usage': None, 'obfuscation': 'gmH8or2Vlb'}, tool_calls=[ToolCallResult(tool_name='search_flights', too

## 🎉 Congratulations!

You've learned how to:
- ✅ Create custom tools for AI agents
- ✅ Build functional agents with LlamaIndex
- ✅ Implement observability and debugging
- ✅ Manage conversation context and memory
- ✅ Integrate MCP-style tools
- ✅ Track agent performance metrics

### Share Your Work:
- Complete the challenge and share your restaurant agent
- Experiment with different tools and domains
- Try integrating real APIs for production use

Happy building! 🚀

## 🎓 Challenge: Build Your Own Agent!

Now it's your turn! Apply what you've learned to create a unique agent.

### Your Mission:
Create a **Restaurant Recommendation Agent** that helps users find and learn about restaurants.

### Requirements:
1. **Create at least 3 custom tools:**
   - `find_restaurants(cuisine: str, city: str) -> str`: Find restaurants by cuisine type
   - `get_restaurant_details(restaurant_name: str) -> str`: Get details like hours, price range
   - `make_reservation(restaurant_name: str, date: str, party_size: int) -> str`: Simulate booking

2. **Add a bonus tool** (get creative!):
   - Menu translator
   - Dietary restriction filter
   - Restaurant reviews summarizer
   - Distance calculator
   - Or your own idea!

3. **Implement:**
   - Agent with all your tools
   - Observability (print tool calls)
   - Context/memory for conversation
   - Test with at least 2 complex queries

4. **Bonus Points:**
   - Make it work with real APIs (like Yelp or Google Places)
   - Add error handling
   - Create a user-friendly interface
   - Implement caching for repeated queries

### Tips:
- Use clear docstrings - the LLM reads them!
- Test each tool individually first
- Start simple, then add complexity
- Use type hints for better tool understanding
- Think about real-world use cases

### Example Test Queries:
- "Find Italian restaurants in Boston and get details about the top one"
- "I'm vegetarian. Find a good restaurant in Seattle and make a reservation for 4 people tomorrow"
- "Recommend a restaurant with outdoor seating, check if they're open now, and book a table"

### Your Solution Space

Write your code below:

In [45]:
#!/usr/bin/env python3
"""
🍽️ Restaurant Recommendation Agent
"""

from datetime import datetime, timedelta
from typing import Dict, List, Tuple, Optional
import random
import json
from functools import lru_cache
import time

In [46]:
# RESTAURANT DATABASE (Simulated)

RESTAURANT_DATABASE = {
    "boston": {
        "italian": [
            {"name": "Bella Italia", "price_range": "$$$", "hours": "11:00-22:00", "cuisine": "Italian", "atmosphere": "Romantic", "rating": 4.8},
            {"name": "Roma Kitchen", "price_range": "$$", "hours": "12:00-23:00", "cuisine": "Italian", "atmosphere": "Casual", "rating": 4.6},
            {"name": "Tuscany Dreams", "price_range": "$$$$", "hours": "17:00-23:00", "cuisine": "Italian", "atmosphere": "Upscale", "rating": 4.9},
        ],
        "japanese": [
            {"name": "Tokyo Sushi Bar", "price_range": "$$$", "hours": "11:30-22:30", "cuisine": "Japanese", "atmosphere": "Modern", "rating": 4.7},
            {"name": "Ramen House", "price_range": "$$", "hours": "12:00-21:00", "cuisine": "Japanese", "atmosphere": "Casual", "rating": 4.5},
        ],
        "vegan": [
            {"name": "Green Haven", "price_range": "$$", "hours": "10:00-21:00", "cuisine": "Vegan", "atmosphere": "Trendy", "rating": 4.7},
        ],
    },
    "seattle": {
        "italian": [
            {"name": "Marco's Trattoria", "price_range": "$$", "hours": "11:00-22:00", "cuisine": "Italian", "atmosphere": "Casual", "rating": 4.6},
            {"name": "Bella Notte", "price_range": "$$$", "hours": "17:00-23:00", "cuisine": "Italian", "atmosphere": "Romantic", "rating": 4.8},
        ],
        "vegetarian": [
            {"name": "Garden Fresh", "price_range": "$$", "hours": "10:30-20:30", "cuisine": "Vegetarian", "atmosphere": "Casual", "rating": 4.5},
            {"name": "Leaf & Root", "price_range": "$$$", "hours": "11:00-21:30", "cuisine": "Vegetarian", "atmosphere": "Trendy", "rating": 4.7},
        ],
        "asian": [
            {"name": "Pan-Asian Fusion", "price_range": "$$$", "hours": "11:30-22:00", "cuisine": "Asian", "atmosphere": "Modern", "rating": 4.6},
        ],
    },
}

ATMOSPHERE_DESCRIPTIONS = {
    "Romantic": "Intimate lighting, soft music, candlelit tables - perfect for dates 🕯️",
    "Casual": "Relaxed vibe, family-friendly, great for groups and quick bites 😊",
    "Upscale": "Fine dining, elegant décor, impeccable service, dress code recommended 🎩",
    "Modern": "Contemporary design, sophisticated, hip crowd, trendy 🎨",
    "Trendy": "Instagram-worthy, popular with millennials, modern twist on classics 📸",
}

REVIEW_SAMPLES = {
    "positive": [
        "Excellent food and great service!",
        "Highly recommend! Best meal I've had all month.",
        "Amazing atmosphere and delicious flavors.",
        "Staff was incredibly attentive. Worth every penny!",
        "Absolutely worth the wait. Fantastic experience!",
    ],
    "neutral": [
        "Good food, average service.",
        "Nice place. Could be better with shorter wait times.",
        "Decent options, prices on the higher side.",
    ],
}

# CACHING

class RestaurantCache:
    """Simple caching system for repeated queries."""
    def __init__(self):
        self.cache: Dict[str, Tuple[str, float]] = {}
        self.hit_count = 0
        self.miss_count = 0

    def get(self, key: str) -> Optional[str]:
        """Get value from cache if not expired (5 minutes)."""
        if key in self.cache:
            value, timestamp = self.cache[key]
            if time.time() - timestamp < 300:  # 5 minutes expiry
                self.hit_count += 1
                return value
            else:
                del self.cache[key]
        self.miss_count += 1
        return None

    def set(self, key: str, value: str):
        """Store value in cache with timestamp."""
        self.cache[key] = (value, time.time())

    def get_stats(self) -> str:
        """Return cache statistics."""
        total = self.hit_count + self.miss_count
        hit_rate = (self.hit_count / total * 100) if total > 0 else 0
        return f"Cache: {self.hit_count} hits, {self.miss_count} misses ({hit_rate:.1f}% hit rate)"


cache = RestaurantCache()


In [47]:
# TOOLS

def find_restaurants(cuisine: str, city: str) -> str:
    """Find restaurants by cuisine type in a specific city.

    This tool searches the restaurant database to find available restaurants
    matching the specified cuisine in the given city. Results include restaurant
    names and basic information.

    Args:
        cuisine: Type of cuisine (e.g., 'Italian', 'Japanese', 'Vegan', 'Vegetarian')
        city: City name (e.g., 'Boston', 'Seattle'). Case-insensitive.

    Returns:
        A string containing matching restaurants with their ratings, or a message
        if no restaurants are found. Formatted for easy reading.

    Raises:
        No exceptions - gracefully handles missing cities/cuisines.
    """
    try:
        # Check cache first
        cache_key = f"find:{cuisine}:{city}"
        cached = cache.get(cache_key)
        if cached:
            return f"[CACHED] {cached}"

        city_lower = city.lower()
        cuisine_lower = cuisine.lower()

        if city_lower not in RESTAURANT_DATABASE:
            result = f"❌ Sorry, we don't have restaurant data for {city} yet."
            cache.set(cache_key, result)
            return result

        restaurants = RESTAURANT_DATABASE[city_lower].get(cuisine_lower, [])

        if not restaurants:
            result = f"❌ No {cuisine} restaurants found in {city}. Try other cuisines!"
            cache.set(cache_key, result)
            return result

        # Format results nicely
        output = f"✅ Found {len(restaurants)} {cuisine} restaurant(s) in {city}:\n"
        for i, rest in enumerate(restaurants, 1):
            output += f"  {i}. {rest['name']} ⭐{rest['rating']} ({rest['price_range']}) - {rest['atmosphere']}\n"

        cache.set(cache_key, output)
        return output

    except Exception as e:
        return f"⚠️ Error finding restaurants: {str(e)}"


def get_restaurant_details(restaurant_name: str) -> str:
    """Get detailed information about a specific restaurant.

    Retrieves comprehensive details about a restaurant including hours of operation,
    price range, cuisine type, and customer reviews.

    Args:
        restaurant_name: Name of the restaurant to look up (case-insensitive)

    Returns:
        A formatted string with restaurant details including hours, price range,
        atmosphere, reviews, and recommendations.
    """
    try:
        # Search for restaurant in database
        for city, cuisines in RESTAURANT_DATABASE.items():
            for cuisine, restaurants in cuisines.items():
                for rest in restaurants:
                    if rest['name'].lower() == restaurant_name.lower():
                        # Found it!
                        reviews = random.sample(REVIEW_SAMPLES["positive"], 1) + \
                                 random.sample(REVIEW_SAMPLES["neutral"], 1)

                        output = f"\n{'='*60}\n"
                        output += f"🏪 {rest['name']}\n"
                        output += f"{'='*60}\n"
                        output += f"📍 Cuisine: {rest['cuisine']}\n"
                        output += f"💰 Price Range: {rest['price_range']}\n"
                        output += f"🕐 Hours: {rest['hours']}\n"
                        output += f"⭐ Rating: {rest['rating']}/5.0\n"
                        output += f"🎭 Atmosphere: {rest['atmosphere']}\n"
                        output += f"\n📝 Sample Reviews:\n"
                        for i, review in enumerate(reviews, 1):
                            output += f"   • {review}\n"
                        output += f"{'='*60}\n"

                        return output

        return f"❌ Restaurant '{restaurant_name}' not found in our database."

    except Exception as e:
        return f"⚠️ Error retrieving details: {str(e)}"


def make_reservation(restaurant_name: str, date: str, party_size: int) -> str:
    """Make a reservation at a restaurant.

    Simulates booking a reservation for a specific date and party size.
    Checks availability and provides confirmation details.

    Args:
        restaurant_name: Name of the restaurant (case-insensitive)
        date: Reservation date in YYYY-MM-DD format (e.g., '2025-12-25')
        party_size: Number of people in the party (1-20)

    Returns:
        A confirmation string with reservation details, reference number, or
        an error message if the restaurant is not found or booking fails.
    """
    try:
        # Validate inputs
        if party_size < 1 or party_size > 20:
            return f"❌ Party size must be between 1 and 20 people."

        # Check if date is in future
        try:
            res_date = datetime.strptime(date, "%Y-%m-%d")
            if res_date < datetime.now():
                return f"❌ Cannot book in the past! Please choose a future date."
        except ValueError:
            return f"❌ Invalid date format. Please use YYYY-MM-DD (e.g., 2025-12-25)"

        # Search for restaurant
        for city, cuisines in RESTAURANT_DATABASE.items():
            for cuisine, restaurants in cuisines.items():
                for rest in restaurants:
                    if rest['name'].lower() == restaurant_name.lower():
                        # Simulate availability check
                        available = random.random() > 0.2  # 80% availability

                        if not available:
                            return f"❌ Sorry, {restaurant_name} is fully booked on {date}.\nTry another date!"

                        # Generate confirmation
                        confirmation_num = f"RES{random.randint(100000, 999999)}"
                        time_slot = f"{random.randint(17, 20)}:{random.choice(['00', '30'])}"

                        output = f"\n{'='*60}\n"
                        output += f"✅ RESERVATION CONFIRMED!\n"
                        output += f"{'='*60}\n"
                        output += f"🏪 Restaurant: {rest['name']}\n"
                        output += f"📅 Date: {date}\n"
                        output += f"🕐 Time: {time_slot}\n"
                        output += f"👥 Party Size: {party_size} people\n"
                        output += f"📞 Confirmation #: {confirmation_num}\n"
                        output += f"💳 Price Range: {rest['price_range']}/person\n"
                        output += f"{'='*60}\n"
                        output += f"✨ Enjoy your meal!\n"

                        return output

        return f"❌ Restaurant '{restaurant_name}' not found in our database."

    except Exception as e:
        return f"⚠️ Error making reservation: {str(e)}"

# Bonus tool
def get_restaurant_atmosphere(restaurant_name: str) -> str:
    """Get detailed atmosphere and ambiance information about a restaurant.

    Provides comprehensive details about the restaurant's atmosphere, vibe,
    and ambiance to help customers decide if it matches their mood and occasion.

    Args:
        restaurant_name: Name of the restaurant (case-insensitive)

    Returns:
        A formatted string describing the atmosphere, decoration style,
        typical crowd, best for (occasions), and recommendations.
    """
    try:
        # Search for restaurant
        for city, cuisines in RESTAURANT_DATABASE.items():
            for cuisine, restaurants in cuisines.items():
                for rest in restaurants:
                    if rest['name'].lower() == restaurant_name.lower():
                        atmosphere = rest['atmosphere']
                        description = ATMOSPHERE_DESCRIPTIONS.get(
                            atmosphere,
                            "Unique and special atmosphere"
                        )

                        recommendations = {
                            "Romantic": ["First dates", "Anniversaries", "Special celebrations"],
                            "Casual": ["Family outings", "Quick lunches", "Group gatherings"],
                            "Upscale": ["Business dinners", "Special occasions", "Fine dining experience"],
                            "Modern": ["Trendy experience", "Instagram-worthy dining", "Tech-savvy crowd"],
                            "Trendy": ["Young professionals", "Social gatherings", "Modern food lovers"],
                        }

                        best_for = recommendations.get(atmosphere, ["General dining"])

                        output = f"\n{'='*60}\n"
                        output += f"🎭 ATMOSPHERE GUIDE: {rest['name']}\n"
                        output += f"{'='*60}\n"
                        output += f"✨ Type: {atmosphere}\n"
                        output += f"📖 Description: {description}\n"
                        output += f"\n👥 Best For:\n"
                        for item in best_for:
                            output += f"   • {item}\n"
                        output += f"\n💡 Pro Tips:\n"
                        output += f"   • Arrive 10-15 minutes early\n"
                        output += f"   • Call ahead for large groups\n"
                        output += f"   • Check dress code recommendations\n"
                        output += f"{'='*60}\n"

                        return output

        return f"❌ Restaurant '{restaurant_name}' not found in our database."

    except Exception as e:
        return f"⚠️ Error retrieving atmosphere info: {str(e)}"


In [48]:
# CREATE AGENT WITH ALL TOOLS

from llama_index.core.agent.workflow import FunctionAgent
from llama_index.core.tools import FunctionTool

# Convert functions to LlamaIndex tools
find_rest_tool = FunctionTool.from_defaults(fn=find_restaurants)
details_tool = FunctionTool.from_defaults(fn=get_restaurant_details)
reservation_tool = FunctionTool.from_defaults(fn=make_reservation)
atmosphere_tool = FunctionTool.from_defaults(fn=get_restaurant_atmosphere)

restaurant_agent = FunctionAgent(
    tools=[find_rest_tool, details_tool, reservation_tool, atmosphere_tool],
    llm=llm,
    verbose=True
)

print("✅ Restaurant Recommendation Agent Created!")
print("📚 Tools Available:")
print("   1. find_restaurants(cuisine, city)")
print("   2. get_restaurant_details(restaurant_name)")
print("   3. make_reservation(restaurant_name, date, party_size)")
print("   4. get_restaurant_atmosphere(restaurant_name) [BONUS TOOL]")


✅ Restaurant Recommendation Agent Created!
📚 Tools Available:
   1. find_restaurants(cuisine, city)
   2. get_restaurant_details(restaurant_name)
   3. make_reservation(restaurant_name, date, party_size)
   4. get_restaurant_atmosphere(restaurant_name) [BONUS TOOL]


In [49]:
# 5 ENHANCED OBSERVABILITY WITH CLI UI

class RestaurantAgentObserver:
    """Advanced observability with beautiful CLI output."""

    def __init__(self):
        self.conversation_history = []
        self.total_tool_calls = 0
        self.start_session = time.time()

    async def run_query(self, query: str, agent: FunctionAgent):
        """Run query with enhanced CLI output."""

        # Header
        print(f"\n{'🔍 NEW QUERY ' + '='*50}")
        print(f"👤 User: {query}")
        print('='*60)

        self.conversation_history.append({"user": query})

        from llama_index.core.agent.workflow import ToolCallResult

        handler = agent.run(query)
        tool_calls_made = []

        # Stream tool calls
        async for event in handler.stream_events():
            if isinstance(event, ToolCallResult):
                self.total_tool_calls += 1
                tool_info = {
                    "tool": event.tool_name,
                    "input": event.tool_kwargs,
                    "output": event.tool_output
                }
                tool_calls_made.append(tool_info)

                print(f"\n🔧 Tool Call #{len(tool_calls_made)}: {event.tool_name}")
                print(f"   📥 Input: {json.dumps(event.tool_kwargs, indent=3)}")
                print(f"   📤 Output:\n{event.tool_output}")
                print('-'*60)

        # Get final result
        result = await handler

        # Summary
        print(f"\n{'✨ FINAL RESPONSE ' + '='*48}")
        print(f"🤖 Agent:\n{result}\n")

        session_duration = time.time() - self.start_session
        print(f"{'📊 SESSION STATS ' + '='*49}")
        print(f"   ⏱️  Session Duration: {session_duration:.1f}s")
        print(f"   🔧 Total Tool Calls (This Session): {self.total_tool_calls}")
        print(f"   💬 Total Queries: {len(self.conversation_history)}")
        print(f"   {cache.get_stats()}")
        print('='*60)

        return result

    def show_summary(self):
        """Display conversation summary."""
        print(f"\n{'📋 CONVERSATION SUMMARY ' + '='*41}")
        for i, item in enumerate(self.conversation_history, 1):
            print(f"  {i}. Q: {item['user'][:60]}...")
        print(f"  Total Queries: {len(self.conversation_history)}")
        print(f"  Total Tool Calls: {self.total_tool_calls}")
        print('='*60)


# Create observer instance
observer = RestaurantAgentObserver()

In [52]:
# Complex Test Queries
print("🎉 RESTAURANT AGENT - INTERACTIVE DEMO")

# Test Query 1: Italian restaurant with details
print("\n[TEST 1] Finding Italian restaurant in Boston with full details")
await observer.run_query(
    "Find Italian restaurants in Boston and get details about the top one",
    restaurant_agent
)


🎉 RESTAURANT AGENT - INTERACTIVE DEMO

[TEST 1] Finding Italian restaurant in Boston with full details

🔍 NEW QUERY ==================================================
👤 User: Find Italian restaurants in Boston and get details about the top one

🔧 Tool Call #1: find_restaurants
   📥 Input: {
   "cuisine": "Italian",
   "city": "Boston"
}
   📤 Output:
[CACHED] ✅ Found 3 Italian restaurant(s) in Boston:
  1. Bella Italia ⭐4.8 ($$$) - Romantic
  2. Roma Kitchen ⭐4.6 ($$) - Casual
  3. Tuscany Dreams ⭐4.9 ($$$$) - Upscale

------------------------------------------------------------

🔧 Tool Call #2: get_restaurant_details
   📥 Input: {
   "restaurant_name": "Tuscany Dreams"
}
   📤 Output:

🏪 Tuscany Dreams
📍 Cuisine: Italian
💰 Price Range: $$$$
🕐 Hours: 17:00-23:00
⭐ Rating: 4.9/5.0
🎭 Atmosphere: Upscale

📝 Sample Reviews:
   • Absolutely worth the wait. Fantastic experience!
   • Good food, average service.

------------------------------------------------------------

✨ FINAL RESPONSE ===

AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='I found three Italian restaurants in Boston. The top-rated one is Tuscany Dreams with a rating of 4.9. It is an upscale Italian restaurant with a price range of $$$$. It operates from 17:00 to 23:00. Reviews mention it as a fantastic experience with good food, though some note average service. Would you like to know more or make a reservation?')]), structured_response=None, current_agent_name='Agent', raw={'id': 'chatcmpl-DDT0ZzvIPshHRO4CGTDk5B0Hp7w4K', 'choices': [{'delta': {'content': None, 'function_call': None, 'refusal': None, 'role': None, 'tool_calls': None}, 'finish_reason': 'stop', 'index': 0, 'logprobs': None, 'content_filter_results': {}}], 'created': 1772102143, 'model': 'gpt-4.1-mini-2025-04-14', 'object': 'chat.completion.chunk', 'service_tier': None, 'system_fingerprint': 'fp_b6f445fc1c', 'usage': None, 'obfuscation': 'QsRtclyds7'},

In [53]:
# Test Query 2: Complex multi-step query with reservation
print("\n[TEST 2] Complex query with dietary preferences and reservation")
await observer.run_query(
    "I'm vegetarian and looking for a good restaurant in Seattle. Find one, tell me about its atmosphere, and make a reservation for 4 people on 2025-12-25",
    restaurant_agent
)


[TEST 2] Complex query with dietary preferences and reservation

🔍 NEW QUERY ==================================================
👤 User: I'm vegetarian and looking for a good restaurant in Seattle. Find one, tell me about its atmosphere, and make a reservation for 4 people on 2025-12-25

🔧 Tool Call #1: find_restaurants
   📥 Input: {
   "cuisine": "Vegetarian",
   "city": "Seattle"
}
   📤 Output:
[CACHED] ✅ Found 2 Vegetarian restaurant(s) in Seattle:
  1. Garden Fresh ⭐4.5 ($$) - Casual
  2. Leaf & Root ⭐4.7 ($$$) - Trendy

------------------------------------------------------------

🔧 Tool Call #2: get_restaurant_atmosphere
   📥 Input: {
   "restaurant_name": "Leaf & Root"
}
   📤 Output:

🎭 ATMOSPHERE GUIDE: Leaf & Root
✨ Type: Trendy
📖 Description: Instagram-worthy, popular with millennials, modern twist on classics 📸

👥 Best For:
   • Young professionals
   • Social gatherings
   • Modern food lovers

💡 Pro Tips:
   • Arrive 10-15 minutes early
   • Call ahead for large groups
   

AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text="I found two vegetarian restaurants in Seattle:\n\n1. Leaf & Root - Trendy, popular with millennials, modern twist on classics. Best for young professionals, social gatherings, and modern food lovers. It's recommended to arrive 10-15 minutes early and call ahead for large groups.\n\n2. Garden Fresh - Casual, relaxed vibe, family-friendly, great for groups and quick bites. Best for family outings, quick lunches, and group gatherings. It's also recommended to arrive 10-15 minutes early and call ahead for large groups.\n\nPlease note that the reservation date you provided, 2025-12-25, is considered in the past by the system. Could you please confirm or provide a future date for the reservation?")]), structured_response=None, current_agent_name='Agent', raw={'id': 'chatcmpl-DDT1Wm21QUXdR38p9tnTELq0aF0OB', 'choices': [{'delta': {'content': None, 'functi

In [54]:

# Test Query 3: Complex query with outdoor seating preference (simulated by atmosphere)
print("\n[TEST 3] Finding a restaurant with outdoor seating (simulated by atmosphere) and booking a table")
await observer.run_query(
    "Recommend a restaurant with outdoor seating, check if they're open now, and book a table",
    restaurant_agent
)

# Show final summary
observer.show_summary()

print("\n" + "-"*60)



[TEST 3] Finding a restaurant with outdoor seating (simulated by atmosphere) and booking a table

🔍 NEW QUERY ==================================================
👤 User: Recommend a restaurant with outdoor seating, check if they're open now, and book a table

✨ FINAL RESPONSE ================================================
🤖 Agent:
To assist you better, could you please provide me with the following details:
1. The city or location where you want the restaurant recommendation.
2. The type of cuisine you prefer (e.g., Italian, Japanese, Vegan, etc.).
3. The date and time you want to book the table.
4. The number of people for the reservation.

With this information, I can find a suitable restaurant with outdoor seating, check its current status, and help you make a reservation.

📊 SESSION STATS =================================================
   ⏱️  Session Duration: 573.4s
   🔧 Total Tool Calls (This Session): 18
   💬 Total Queries: 8
   Cache: 2 hits, 4 misses (33.3% hit rate)

📋 CO